In [ ]:
import torch
import os
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")
else:
    print("❌ GPU NOT AVAILABLE")

In [ ]:
!pip install -q -U \
    diffusers \
    transformers \
    accelerate \
    safetensors \
    sentencepiece \
    protobuf \
    mediapipe \
    opencv-python-headless \
    scipy \
    matplotlib \
    pillow

In [ ]:
import diffusers
import transformers
import accelerate
import mediapipe
import cv2
import scipy
import PIL

print("Diffusers   :", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate  :", accelerate.__version__)
print("MediaPipe   :", mediapipe.__version__)
print("OpenCV      :", cv2.__version__)
print("SciPy       :", scipy.__version__)
print("Pillow      :", PIL.__version__)

In [ ]:
from pathlib import Path

BASE = Path("/kaggle/working/rhinoplasty_dataset")

directories = [
    BASE / "patients",
    BASE / "models",
    BASE / "validation",
    BASE / "logs",
    BASE / "batches",
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE)
print("\\nCreated:")
for directory in directories:
    print("✓", directory)


In [ ]:
import json
from pathlib import Path

# Optional input contract:
# - In a future Kaggle integration, attach a Kaggle Dataset containing manifest.json.
# - This cell searches Kaggle input mounts for that file.
# - If none is found, the notebook remains usable in manual mode.
BASE = Path("/kaggle/working/rhinoplasty_dataset")
MANIFEST_SEARCH_ROOT = Path("/kaggle/input")
manifest_candidates = (
    sorted(MANIFEST_SEARCH_ROOT.glob("**/manifest.json"))
    if MANIFEST_SEARCH_ROOT.exists()
    else []
)

MANIFEST_PATH = manifest_candidates[0] if manifest_candidates else None

if MANIFEST_PATH:
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        GENERATION_MANIFEST = json.load(f)
    print("Manifest loaded:", MANIFEST_PATH)
    print("Batch ID:", GENERATION_MANIFEST.get("batch_id", "batch-001"))
else:
    GENERATION_MANIFEST = None
    print("No manifest.json found; manual testing mode is active.")

if GENERATION_MANIFEST is None:
    raise RuntimeError("Required manifest.json was not found in /kaggle/input.")

# Manual defaults are used only when no manifest is available.
if GENERATION_MANIFEST is None:
    GENERATION_SETTINGS = {
        "batch_id": "manual-batch",
        "requested_count": 5,
        "prompts": None,  # Uses P01_PROMPT from the generation cell below.
        "seeds": [0, 1, 2, 3, 4],
        "width": 384,
        "height": 384,
        "num_inference_steps": 12,
        "guidance_scale": 1.0,
    }
else:
    GENERATION_SETTINGS = {
        "batch_id": str(GENERATION_MANIFEST.get("batch_id", "batch-001")),
        "requested_count": int(GENERATION_MANIFEST.get("requested_count", 0)),
        "prompts": GENERATION_MANIFEST.get("prompts"),
        "seeds": GENERATION_MANIFEST.get("seeds", []),
        "width": int(GENERATION_MANIFEST.get("width", GENERATION_MANIFEST.get("image_width", 384))),
        "height": int(GENERATION_MANIFEST.get("height", GENERATION_MANIFEST.get("image_height", 384))),
        "num_inference_steps": int(
            GENERATION_MANIFEST.get("provider_config", {}).get(
                "num_inference_steps",
                GENERATION_MANIFEST.get("num_inference_steps", 12),
            )
        ),
        "guidance_scale": float(
            GENERATION_MANIFEST.get("provider_config", {}).get(
                "guidance_scale",
                GENERATION_MANIFEST.get("guidance_scale", 1.0),
            )
        ),
    }

if GENERATION_SETTINGS["requested_count"] < 1:
    raise ValueError("Manifest requested_count must be at least 1.")

if len(GENERATION_SETTINGS["seeds"]) != GENERATION_SETTINGS["requested_count"]:
    raise ValueError(
        "Manifest must contain exactly one seed per requested image. "
        f"Requested {GENERATION_SETTINGS['requested_count']}, "
        f"received {len(GENERATION_SETTINGS['seeds'])} seeds."
    )

prompts_value = GENERATION_SETTINGS["prompts"]
if prompts_value is not None and not (
    isinstance(prompts_value, list)
    and len(prompts_value) == GENERATION_SETTINGS["requested_count"]
    and all(isinstance(p, str) and p.strip() for p in prompts_value)
):
    raise ValueError("Manifest prompts must be a list of non-empty strings, one per image.")

batch_id = "".join(
    ch if ch.isalnum() or ch in "-_" else "-"
    for ch in GENERATION_SETTINGS["batch_id"]
).strip("-") or "batch-001"

BATCH_DIR = BASE / "batches" / batch_id
BATCH_IMAGE_DIR = BATCH_DIR / "images"
BATCH_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("Batch:", batch_id)
print("Requested images:", GENERATION_SETTINGS["requested_count"])
print("Dimensions:", f'{GENERATION_SETTINGS["width"]}x{GENERATION_SETTINGS["height"]}')
print("Output directory:", BATCH_DIR)


In [ ]:
import importlib.util

print("TorchAO installed:", importlib.util.find_spec("torchao") is not None)

from diffusers import Flux2KleinPipeline

print("FLUX.2 Klein import successful!")

In [ ]:
import torch
from diffusers import Flux2KleinPipeline

MODEL_ID = "black-forest-labs/FLUX.2-klein-4B"

pipe = Flux2KleinPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

pipe.enable_model_cpu_offload()

print("FLUX.2 Klein 4B loaded successfully!")

In [ ]:
import json
from pathlib import Path

# Optional input contract:
# - In a future Kaggle integration, attach a Kaggle Dataset containing manifest.json.
# - This cell searches Kaggle input mounts for that file.
# - If none is found, the notebook remains usable in manual mode.
BASE = Path("/kaggle/working/rhinoplasty_dataset")
MANIFEST_SEARCH_ROOT = Path("/kaggle/input")
manifest_candidates = (
    sorted(MANIFEST_SEARCH_ROOT.glob("**/manifest.json"))
    if MANIFEST_SEARCH_ROOT.exists()
    else []
)

MANIFEST_PATH = manifest_candidates[0] if manifest_candidates else None

if MANIFEST_PATH:
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        GENERATION_MANIFEST = json.load(f)
    print("Manifest loaded:", MANIFEST_PATH)
    print("Batch ID:", GENERATION_MANIFEST.get("batch_id", "batch-001"))
else:
    GENERATION_MANIFEST = None
    print("No manifest.json found; manual testing mode is active.")

if GENERATION_MANIFEST is None:
    raise RuntimeError("Required manifest.json was not found in /kaggle/input.")

# Manual defaults are used only when no manifest is available.
if GENERATION_MANIFEST is None:
    GENERATION_SETTINGS = {
        "batch_id": "manual-batch",
        "requested_count": 5,
        "prompts": None,  # Uses P01_PROMPT from the generation cell below.
        "seeds": [0, 1, 2, 3, 4],
        "width": 384,
        "height": 384,
        "num_inference_steps": 12,
        "guidance_scale": 1.0,
    }
else:
    GENERATION_SETTINGS = {
        "batch_id": str(GENERATION_MANIFEST.get("batch_id", "batch-001")),
        "requested_count": int(GENERATION_MANIFEST.get("requested_count", 0)),
        "prompts": GENERATION_MANIFEST.get("prompts"),
        "seeds": GENERATION_MANIFEST.get("seeds", []),
        "width": int(GENERATION_MANIFEST.get("width", GENERATION_MANIFEST.get("image_width", 384))),
        "height": int(GENERATION_MANIFEST.get("height", GENERATION_MANIFEST.get("image_height", 384))),
        "num_inference_steps": int(
            GENERATION_MANIFEST.get("provider_config", {}).get(
                "num_inference_steps",
                GENERATION_MANIFEST.get("num_inference_steps", 12),
            )
        ),
        "guidance_scale": float(
            GENERATION_MANIFEST.get("provider_config", {}).get(
                "guidance_scale",
                GENERATION_MANIFEST.get("guidance_scale", 1.0),
            )
        ),
    }

if GENERATION_SETTINGS["requested_count"] < 1:
    raise ValueError("Manifest requested_count must be at least 1.")

if len(GENERATION_SETTINGS["seeds"]) != GENERATION_SETTINGS["requested_count"]:
    raise ValueError(
        "Manifest must contain exactly one seed per requested image. "
        f"Requested {GENERATION_SETTINGS['requested_count']}, "
        f"received {len(GENERATION_SETTINGS['seeds'])} seeds."
    )

prompts_value = GENERATION_SETTINGS["prompts"]
if prompts_value is not None and not (
    isinstance(prompts_value, list)
    and len(prompts_value) == GENERATION_SETTINGS["requested_count"]
    and all(isinstance(p, str) and p.strip() for p in prompts_value)
):
    raise ValueError("Manifest prompts must be a list of non-empty strings, one per image.")

batch_id = "".join(
    ch if ch.isalnum() or ch in "-_" else "-"
    for ch in GENERATION_SETTINGS["batch_id"]
).strip("-") or "batch-001"

BATCH_DIR = BASE / "batches" / batch_id
BATCH_IMAGE_DIR = BATCH_DIR / "images"
BATCH_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("Batch:", batch_id)
print("Requested images:", GENERATION_SETTINGS["requested_count"])
print("Dimensions:", f'{GENERATION_SETTINGS["width"]}x{GENERATION_SETTINGS["height"]}')
print("Output directory:", BATCH_DIR)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import torch

BASE = Path("/kaggle/working/rhinoplasty_dataset")
P01_DIR = BASE / "patients" / "P01"
CANDIDATE_DIR = P01_DIR / "normal" / "candidates"
CANDIDATE_DIR.mkdir(parents=True, exist_ok=True)
BATCH_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

P01_PROMPT = """
Photorealistic clinical-style portrait photograph of a fictional
adult East Asian person, approximately 25 to 35 years old,
front-facing straight-on view, looking directly at the camera,
neutral relaxed facial expression.

Natural normal nose with realistic anatomical proportions,
no obvious nasal hump, no obvious nasal deviation,
natural nasal bridge and tip.

Natural realistic skin texture with subtle pores,
realistic facial anatomy, natural eyes and lips,
stable simple hairstyle.

No glasses, no sunglasses, no hat, no jewelry,
no facial accessories, no heavy makeup.

Even neutral studio lighting, soft natural shadows,
plain neutral background, single person,
centered face, realistic photography,
sharp facial details, no artistic stylization.
"""

requested_count = GENERATION_SETTINGS["requested_count"]
seeds = GENERATION_SETTINGS["seeds"]
prompts = GENERATION_SETTINGS["prompts"]
width = GENERATION_SETTINGS["width"]
height = GENERATION_SETTINGS["height"]
num_inference_steps = GENERATION_SETTINGS["num_inference_steps"]
guidance_scale = GENERATION_SETTINGS["guidance_scale"]

batch_records = []
print(f"Generating {requested_count} independent face image(s) for batch {batch_id}...")

for index in range(requested_count):
    seed = int(seeds[index])
    prompt = prompts[index] if prompts is not None else P01_PROMPT
    filename = f"image_{index + 1:04d}.png"
    batch_path = BATCH_IMAGE_DIR / filename

    print(f"Generating image {index + 1}/{requested_count} | seed={seed}")
    record = {
        "index": index + 1,
        "filename": filename,
        "prompt": prompt,
        "seed": seed,
        "width": width,
        "height": height,
        "status": "FAILED",
        "error": None,
    }

    try:
        generator = torch.Generator(device="cpu").manual_seed(seed)
        with torch.inference_mode():
            result = pipe(
                prompt=prompt,
                height=height,
                width=width,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
                generator=generator,
            )

        image = result.images[0]
        image.save(batch_path, format="PNG")

        # Keep the old candidate naming convention available for the later,
        # unchanged multi-view cells. This compatibility copy is not the batch source of truth.
        legacy_path = CANDIDATE_DIR / f"P01_candidate_{seed}.png"
        image.save(legacy_path, format="PNG")

        record.update({
            "status": "GENERATED",
            "path": str(batch_path),
            "legacy_candidate_path": str(legacy_path),
            "actual_width": image.width,
            "actual_height": image.height,
        })
        print("✓ Saved:", batch_path)
    except Exception as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
        print(f"✗ Failed image {index + 1}: {record['error']}")

    batch_records.append(record)

generated_count = sum(r["status"] == "GENERATED" for r in batch_records)
failed_count = requested_count - generated_count
metadata_path = BATCH_DIR / "metadata.json"
summary_path = BATCH_DIR / "batch_summary.json"

metadata_document = {
    "batch_id": batch_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "generation_mode": "manifest" if GENERATION_MANIFEST is not None else "manual",
    "requested_count": requested_count,
    "generated_count": generated_count,
    "failed_count": failed_count,
    "images": batch_records,
}
metadata_path.write_text(json.dumps(metadata_document, indent=2), encoding="utf-8")

summary_document = {
    "job_id": int(GENERATION_MANIFEST.get("job_id", 0)) if GENERATION_MANIFEST else 0,
    "batch_id": batch_id,
    "requested_count": requested_count,
    "generated_count": generated_count,
    "successful_count": generated_count,
    "failed_count": failed_count,
    "image_width": width,
    "image_height": height,
    "output_directory": str(BATCH_DIR),
    "metadata_file": str(metadata_path),
    "images": [
        {
            "filename": r["filename"],
            "seed": r["seed"],
            "prompt": r["prompt"],
            "width": r["width"],
            "height": r["height"],
            "status": r["status"],
            "path": r.get("path"),
            "error": r.get("error"),
        }
        for r in batch_records
    ],
}
summary_path.write_text(json.dumps(summary_document, indent=2), encoding="utf-8")

print("\\nBatch finished.")
print(f"Requested: {requested_count} | Generated: {generated_count} | Failed: {failed_count}")
print("Metadata:", metadata_path)
print("Summary:", summary_path)

if failed_count:
    print("WARNING: Some images failed; inspect batch_summary.json and metadata.json.")
